In [0]:
from pyspark.sql.functions import col
df_licitaciones = spark.table(
    "chilecompra.silver.licitaciones"
)

In [0]:
df_licitaciones_validas = (
    df_licitaciones
    .filter(
        col("fecha_cierre").isNotNull()
        )
)

In [0]:
from pyspark.sql.functions import (
    year,
    month,
    countDistinct,
    sum,
    avg
)

df_licitaciones_estado = (
    df_licitaciones_validas
    .groupBy(
        year(col("fecha_cierre")).alias("anio"),
        month(col("fecha_cierre")).alias("mes"),
        col("codigo_estado")
    )
    .agg(
        countDistinct("codigo_externo").alias("cantidad_licitaciones")
    )
    .orderBy(
        "anio",
        "mes",
        "codigo_estado"
    )
)

In [0]:
total_rows = df_licitaciones_estado.count()

null_keys = (
    df_licitaciones_estado
    .filter(
        col("anio").isNull()
        | col("mes").isNull()
        | col("codigo_estado").isNull()
    )
    .count()
)

distinct_keys = (
    df_licitaciones_estado
    .select(
        "anio",
        "mes",
        "codigo_estado"
    )
    .distinct()
    .count()
)

invalid_counts = (
    df_licitaciones_estado
    .filter(col("cantidad_licitaciones") <= 0)
    .count()
)

if total_rows == 0:
    raise ValueError(
        "DQ FAILED: licitaciones aggregation produced 0 rows"
    )

if null_keys > 0:
    raise ValueError(
        f"DQ FAILED: {null_keys} rows have NULL business keys"
    )

if total_rows != distinct_keys:
    raise ValueError(
        "DQ FAILED: duplicated year/month/state keys"
    )

if invalid_counts > 0:
    raise ValueError(
        f"DQ FAILED: {invalid_counts} rows have invalid licitacion counts"
    )

print(
    f"Pre-write licitaciones Gold DQ passed: {total_rows} rows"
)

In [0]:
target_table = "chilecompra.gold.licitaciones_por_mes_cierre_estado"

(
    df_licitaciones_estado
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

In [0]:
df_gold_licitaciones = spark.table(target_table)

gold_rows = df_gold_licitaciones.count()

gold_distinct_keys = (
    df_gold_licitaciones
    .select(
        "anio",
        "mes",
        "codigo_estado"
    )
    .distinct()
    .count()
)

if gold_rows != total_rows:
    raise ValueError(
        "DQ FAILED: Gold row count does not match source aggregation"
    )

if gold_rows != gold_distinct_keys:
    raise ValueError(
        "DQ FAILED: duplicated year/month/state keys in Gold"
    )

print(
    f"Gold licitaciones_por_mes_cierre_estado DQ passed: "
    f"{gold_rows} rows"
)